# PCB 결함 검????YOLOv8 K-Fold 교차 검�?Colab ?�습 ?�트�?n
**Team Convex | ATI ?�턴 ?�로?�트**

???�트북�? `scripts/run_kfold.bat`�?**?�일??K-Fold ?�습 ?�름**??Google Colab?�서 ?�행?�며,
Google Drive ?�동?�로 **?�션???�겨???�른 PC/Colab?�서 ?�어?�습**??가?�합?�다.

---

## ?�행 ??준�?n
1. **GitHub Token ?�록** (Private Repo ?�근??
   - Colab ?�쪽 ?�이?�바 ?�� ?�이�??�릭
   - `GITHUB_TOKEN` ?�름?�로 Personal Access Token(PAT) 추�?
   - PAT ?�성: GitHub ??Settings ??Developer settings ??Personal access tokens
   - ?�요 권한: `Contents: Read`

2. **dataset.zip Drive ?�로??*
   - Google Drive `MyDrive/pcb-defect-inspection/` ?�더??`dataset.zip` ?�로??n
3. **?��????�형 ?�인**
   - 메뉴: ?��??????��????�형 변�???**T4 GPU** ?�택

---

## ?�습 ?�름 (run_kfold.bat�??�일)

| ?� | ?�계 | 비고 |
|----|------|------|
| ??| ?�경 ?�정 (GPU ?�인) | 경로 ?�수 ?�의 |
| ??| Google Drive 마운??| |
| ??| Repo ?�론 & ?�키지 ?�치 | Private repo 지??|
| ??| ?�정 ?�인 | config.yaml ?�라미터 출력 |
| ??| ?�처�?| ?��? ?�료 ???�동 ?�킵 |
| ??| ?�어?�습 체크 | Drive??fold 체크?�인???�동 감�? |
| ??| K-Fold ?�습 ?�행 | --resume ?�동 ?�용 |
| ??| Drive ?�기??| ?�션 종료 ???�수! |
| ??| 결과 ?�각??| Fold�?결과 ?�약 (?�택 ?�항) |

## ???�경 ?�정

> `DRIVE_PROJECT_DIR`???�제 Drive ?�더?� ?�르�??�정 ???�행?�세??

In [ ]:
# ============================================================
# ???�경 ?�정 ???�래 ?�수�??�인 ???�행?�세??n# ============================================================

# --- GitHub ?�정 ---
GITHUB_REPO   = "june0922/pcb-defect-inspection"
GITHUB_BRANCH = "main"  # ?�용??브랜치명
PROJECT_DIR   = "/content/pcb-defect-inspection"

# --- Google Drive 경로 ---
# dataset.zip???�로?�한 Drive ?�더 경로 (?�요???�정)
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/pcb-defect-inspection"
DRIVE_DATASET_ZIP = f"{DRIVE_PROJECT_DIR}/dataset.zip"

# --- GPU ?�인 ---
import subprocess
try:
    r = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        capture_output=True, text=True
    )
    if r.returncode == 0:
        print(f"??GPU: {r.stdout.strip()}")
    else:
        print("?�️  GPU ?�음 ???��????�형??'GPU'�?변경하?�요.")
        print("   메뉴: ?��???> ?��????�형 변�?> T4 GPU ?�택")
except FileNotFoundError:
    print("?�️  GPU ?�음 (nvidia-smi 명령?��? 찾을 ???�음) ???��????�형??'GPU'�?변경하?�요.")
    print("   메뉴: ?��???> ?��????�형 변�?> T4 GPU ?�택")

print(f"\n?�로?�트 ?�렉?�리 : {PROJECT_DIR}")
print(f"Drive ?�로?�트 ?�더: {DRIVE_PROJECT_DIR}")
print(f"Drive dataset.zip  : {DRIVE_DATASET_ZIP}")

## ??Google Drive 마운??


In [ ]:
# ============================================================
# ??Google Drive 마운??n# ============================================================
from google.colab import drive
import os

drive.mount('/content/drive')

os.makedirs(DRIVE_PROJECT_DIR, exist_ok=True)
print("??Drive 마운???�료")
print(f"Drive ?�로?�트 ?�더: {DRIVE_PROJECT_DIR}")
contents = os.listdir(DRIVE_PROJECT_DIR)
print(f"?�재 ?�용: {contents if contents else '(비어 ?�음)'}")

## ??Repo ?�론 & ?�키지 ?�치 (Private Repo)

> **?�전 준�?*: Colab ?�쪽 ?�� ?�이�???`GITHUB_TOKEN` ?�록 ?�요

In [ ]:
# ============================================================
# ??Repo ?�론 & ?�키지 ?�치 (Private Repo)
#    보안: subprocess�??�론???�큰??출력???�출?��? ?�습?�다.
# ============================================================
from google.colab import userdata
import subprocess, os

# --- GitHub Token 로드 ---
try:
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
    print("??GitHub Token 로드??(Colab Secrets)")
except Exception:
    GITHUB_TOKEN = ""
    print("??GITHUB_TOKEN??찾을 ???�습?�다.")
    print("   ?�결: ?�쪽 ?�이?�바 ??�??�릭 ??'GITHUB_TOKEN' 추�?")
    raise RuntimeError("GITHUB_TOKEN???�요?�니??")

# --- Repo ?�론 ?�는 ?�데?�트 ---
repo_url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_REPO}.git"

if not os.path.exists(PROJECT_DIR):
    print(f"'{GITHUB_REPO}' ?�론 �?.. (?�정 ?�더 ?�외 ?�정 ?�용)")
    result = subprocess.run(
        ["git", "clone", "--no-checkout", "--branch", GITHUB_BRANCH, repo_url, PROJECT_DIR],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        err = result.stderr.replace(GITHUB_TOKEN, "***")
        raise RuntimeError(f"?�론 ?�패:\n{err}")
    
    subprocess.run(["git", "-C", PROJECT_DIR, "config", "core.sparseCheckout", "true"], check=True)
    sparse_file = os.path.join(PROJECT_DIR, ".git", "info", "sparse-checkout")
    with open(sparse_file, "w", encoding="utf-8") as f:
        f.write("/*\n")
        f.write("!/.agents/\n")
        f.write("!/app/\n")
        f.write("!/runs/\n")
        f.write("!/web_*/\n")
        f.write("!/weights/\n")
    
    subprocess.run(["git", "-C", PROJECT_DIR, "checkout", GITHUB_BRANCH], capture_output=True, text=True, check=True)
    print(f"??Repo ?�론 ?�료: {PROJECT_DIR}")
else:
    print(f"?�️  ?��? ?�론??({PROJECT_DIR}) ??최신?�로 ?�데?�트?�니??")
    subprocess.run(
        ["git", "-C", PROJECT_DIR, "remote", "set-url", "origin", repo_url],
        capture_output=True
    )
    result = subprocess.run(
        ["git", "-C", PROJECT_DIR, "pull"],
        capture_output=True, text=True
    )
    print(result.stdout.strip() or "?��? 최신 ?�태?�니??")

# --- ?�업 ?�렉?�리 ?�동 ---
%cd {PROJECT_DIR}
print(f"\n?�업 ?�렉?�리: {os.getcwd()}")

# --- ?�키지 ?�치 ---
print("\n?�키지 ?�치 �?..")
!pip install -q -r requirements.txt
print("???�키지 ?�치 ?�료")

## ???�정 ?�인

> ?�래 ?��? ?�인?�고 ?�정??맞으�??�음 ?��?진행?�세??  
> 변경이 ?�요?�면 `config.yaml`??직접 ?�정 ?????�???�시 ?�행?�세??  
> **K-Fold 관???�라미터**: `kfold.k` (기본 5), `kfold.random_state` (기본 42)

In [ ]:
# ============================================================
# ??config.yaml ??env=colab ?�치 �??�정 출력
#    (run_kfold.bat??show_config.py + env ?�환 ??��)
# ============================================================
import yaml, sys, os
sys.path.insert(0, "src")

with open("config.yaml", "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

# env�?colab?�로 ?�환
cfg["env"] = "colab"
with open("config.yaml", "w", encoding="utf-8") as f:
    yaml.dump(cfg, f, allow_unicode=True)

# ?�정 출력 (show_config.py?� ?�일???�이�??�식)
def print_dict(d, prefix=''):
    for k, v in d.items():
        if isinstance(v, dict):
            print_dict(v, prefix + k + '.')
        else:
            v_str = ', '.join(map(str, v)) if isinstance(v, list) else str(v)
            print(f"| {prefix+k:<33} | {v_str:<25} |")

print("+" + "-"*35 + "+" + "-"*27 + "+")
print(f"| {'Parameter':<33} | {'Value':<25} |")
print("+" + "-"*35 + "+" + "-"*27 + "+")
print_dict(cfg)
print("+" + "-"*35 + "+" + "-"*27 + "+")
print()

# K-Fold ?�라미터 별도 강조 출력
kf = cfg.get("kfold", {})
print(f"?�� K-Fold ?�정: k={kf.get('k', 5)}, random_state={kf.get('random_state', 42)}")
print()
print("?�️  ???�정???�인 ?? 문제?�으�??�음 ?�(???�처�????�행?�세??")

## ???�처�?n
- `preprocessed_data/` ?�더가 ?��? ?�으�?**?�동?�로 건너?�니??*
- 강제 ?�처�? `FORCE_PREPROCESS = True` �?변�????�행

> **?�전 준�?*: Drive `pcb-defect-inspection/dataset.zip` ?�로???�요

In [ ]:
# ============================================================
# ???�이???�처�?(run_kfold.bat??preprocess ?�택 ?�계)
#    preprocessed_data/ 가 ?�으�??�킵, ?�으�?Drive?�서 zip 복사 ???�행
# ============================================================
import sys, os, shutil
from pathlib import Path
sys.path.insert(0, "src")
from utils import load_config, get_paths

FORCE_PREPROCESS = False  # True�?변경하�?강제 ?�처�?n
cfg = load_config("config.yaml")
paths = get_paths(cfg)
processed_dir = paths["processed"]
train_img_dir = processed_dir / "images" / "train"

already_done = train_img_dir.exists() and any(train_img_dir.glob("*.jpg"))

if already_done and not FORCE_PREPROCESS:
    n_train = len(list((processed_dir / "images" / "train").glob("*.jpg")))
    n_val   = len(list((processed_dir / "images" / "val").glob("*.jpg")))
    n_test  = len(list((processed_dir / "images" / "test").glob("*.jpg")))
    print("?�️  ?��? ?�처리된 ?�이?��? ?�습?�다 ??건너?�니??")
    print(f"   train: {n_train}??/ val: {n_val}??/ test: {n_test}??")
    print("   강제 ?�처�? FORCE_PREPROCESS = True �?변�????�실??")
else:
    # --- dataset.zip 준�?---
    local_zip = Path(PROJECT_DIR) / "dataset.zip"
    if not local_zip.exists():
        if os.path.exists(DRIVE_DATASET_ZIP):
            size_mb = os.path.getsize(DRIVE_DATASET_ZIP) / 1e6
            print(f"Drive?�서 dataset.zip 복사 �?.. ({size_mb:.1f} MB)")
            shutil.copy2(DRIVE_DATASET_ZIP, local_zip)
            print("??복사 ?�료")
        else:
            msg = (
                "dataset.zip??찾을 ???�습?�다.\n"
                f"Drive 경로: {DRIVE_DATASET_ZIP}\n"
                f"Local 경로: {local_zip}\n"
                "Google Drive??pcb-defect-inspection/ ?�더??dataset.zip???�로?�하?�요."
            )
            raise FileNotFoundError(msg)
    else:
        size_mb = local_zip.stat().st_size / 1e6
        print(f"?�️  dataset.zip 로컬??존재 ({size_mb:.1f} MB)")

    # --- ?�처�??�행 ---
    print("\n?�처�??�작...")
    !python src/preprocess.py --config config.yaml
    print("\n???�처�??�료")

## ???�어?�습(Resume) 체크

Drive??K-Fold 체크?�인?��? ?�동?�로 감�????�어?�습 ?��?�?결정?�니??

- **?�료??fold** (`weights/best_fold_N.pt` 존재) ??**?�동 ?�킵**
- **중단??fold** (`runs/kfold/fold_N/weights/last.pt` 존재) ??**last.pt?�서 ?�어?�습**
- **미시??fold** ??**처음부???�습**

| 변??| ?�작 |
|------|------|
| `FORCE_RESUME = None` | **?�동 감�?** (기본�? 권장) |
| `FORCE_RESUME = True` | 체크?�인???�어??강제 ?�어?�습 ?�도 |
| `FORCE_RESUME = False` | 체크?�인???�어??처음부???�습 |

In [ ]:
# ============================================================
# ???�어?�습(Resume) & Drive ?�시�??�기???�정
#    runs ?�더�?Google Drive???�볼�?링크�??�결?�여
#    ?�습 �??�션???�겨??가중치(last.pt)가 Drive???�시�??�?�되?�록 ??
# ============================================================
import os, shutil, yaml
from pathlib import Path
sys.path.insert(0, "src")
from utils import load_config, get_paths

FORCE_RESUME = None  # None=?�동, True=강제 ?�어?�습, False=강제 처음부??n
DRIVE_RUNS = f"{DRIVE_PROJECT_DIR}/runs"
LOCAL_RUNS = f"{PROJECT_DIR}/runs"

# 1. Drive ?�시�??�?�을 ?�한 ?�볼�?링크(Symlink) ?�정
os.makedirs(DRIVE_RUNS, exist_ok=True)
if os.path.exists(LOCAL_RUNS) and not os.path.islink(LOCAL_RUNS):
    shutil.rmtree(LOCAL_RUNS)  # 기존 로컬 ?�더 ?�거
if not os.path.exists(LOCAL_RUNS):
    os.symlink(DRIVE_RUNS, LOCAL_RUNS)  # 로컬 runs -> Drive runs ?�결
print("[Drive ?�기?? 로컬 runs/ ?�더가 Google Drive???�시�??�결?�었?�니??")

# 2. K-Fold 체크?�인???�태 ?�캔
cfg = load_config("config.yaml")
paths = get_paths(cfg)
kf_cfg = cfg.get("kfold", {"k": 5, "random_state": 42})
K = kf_cfg.get("k", 5)

print(f"\n[K-Fold 체크?�인???�태] k={K}")
print("-" * 52)

any_checkpoint = False
for fold_num in range(1, K + 1):
    best_pt = paths["weights"] / f"best_fold_{fold_num}.pt"
    last_pt = paths["runs"] / "kfold" / f"fold_{fold_num}" / "weights" / "last.pt"
    if best_pt.exists():
        status = f"???�료 ({best_pt.name})"
        any_checkpoint = True
    elif last_pt.exists():
        status = f"?�� 중단????last.pt?�서 ?�어?�습"
        any_checkpoint = True
    else:
        status = "?? 미시??"
    print(f"  Fold {fold_num}/{K}: {status}")

print("-" * 52)

# 3. RESUME ?�래�?결정
if FORCE_RESUME is not None:
    RESUME = FORCE_RESUME
    status = '?�어?�습 (강제)' if RESUME else '처음부???�습 (강제)'
    print(f"\nFORCE_RESUME = {FORCE_RESUME} ??{status}")
elif any_checkpoint:
    print(f"\n??중단??fold 발견 ???�어?�습 ?�동 ?�성??")
    RESUME = True
else:
    print("\n?�️  체크?�인???�음 ??처음부???�습?�니??")
    RESUME = False

print()
print("=" * 42)
print(f"  RESUME = {RESUME}")
if RESUME:
    print("  ?�� ?�음 ?�: '--resume' ?�래그로 ?�어?�습")
    print("  처음부???�시 ?�려�? FORCE_RESUME = False")
else:
    print("  ?? ?�음 ?�: ??K-Fold ?�습 ?�작")
    print("  ?�어?�습???�려�? FORCE_RESUME = True")
print("=" * 42)


## ??K-Fold ?�습 ?�행

> ???�??`RESUME` 값에 ?�라 `--resume` ?�래그�? ?�동 ?�용?�니??  
> ?�습 �??�션???�겨?? ?�음 ?�션?�서 ???�을 ?�실?�하�??�동 ?�어?�습?�니??  
>
> **?�??경로**:
> - Fold�?결과: `runs/kfold/fold_N/`
> - Fold�?최적 가중치: `weights/best_fold_N.pt`

In [ ]:
# ============================================================
# ??K-Fold ?�습 ?�행 (run_kfold.bat??python src/train_kfold.py ??��)
# ============================================================
print("=" * 52)
if RESUME:
    print("  ?�� K-Fold ?�어?�습 ?�작 (--resume)")
else:
    print("  ?? K-Fold ???�습 ?�작")
print("=" * 52)
print()

if RESUME:
    !python src/train_kfold.py --config config.yaml --resume
else:
    !python src/train_kfold.py --config config.yaml

print()
print("=" * 52)
print("  K-Fold ?�습 종료 (?�는 ?�션 중단)")
print("  ??반드???�음 ?� ??Drive ?�기?��? ?�행?�세??")
print("=" * 52)

## ??Drive ?�기??(?�택/?�중?�전)

> ???�?�서 ?�볼�?링크�??�정?�다�????�???�어??**가중치(last.pt/best_fold_N.pt)??Drive???�전**?�니??  
> 
> ?�만 최종?�으�?`weights/best_fold_N.pt` ?�일??복사?�두�??�해 ?�행?�두�?좋습?�다.

In [ ]:
import shutil, os
from pathlib import Path

def sync_dir_to_drive(local_dir: str, drive_dir: str) -> int:
    local_path = Path(local_dir)
    drive_path = Path(drive_dir)
    if not local_path.exists():
        print(f"  ?�️  ?�음 (건너?�): {local_dir}")
        return 0

    # If local_dir is a symlink to drive_dir, skip copying files within it
    # as they are already synchronized.
    if local_path.is_symlink() and local_path.resolve() == drive_path.resolve():
        print(f"  ?�️  '{local_dir}'???��? Drive???�볼�?링크?�어 ?�습?�다 ??복사 건너?�.")
        return 0

    drive_path.mkdir(parents=True, exist_ok=True)
    count = 0
    for src in local_path.rglob("*"):
        if src.is_file():
            dst = drive_path / src.relative_to(local_path)
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dst)
            count += 1
    return count

print("Drive ?�기??�?..\n")

# runs/kfold/ ?�기??(last.pt, best.pt, results.png ??fold�?결과)
n1 = sync_dir_to_drive(
    f"{PROJECT_DIR}/runs",
    f"{DRIVE_PROJECT_DIR}/runs"
)
print(f"  ??runs/    ??Drive ({n1}�??�일)")

# weights/ ?�기??(best_fold_N.pt)
n2 = sync_dir_to_drive(
    f"{PROJECT_DIR}/weights",
    f"{DRIVE_PROJECT_DIR}/weights"
)
print(f"  ??weights/ ??Drive ({n2}�??�일)")

print()
print(f"?�� ?�기???�료 ??{DRIVE_PROJECT_DIR}")
print("   ?�음 ?�션?�서 ?�어?�습 ??Drive??last.pt / best_fold_N.pt가 ?�동?�로 복원?�니??")

## ??결과 ?�각??(?�택 ?�항)

K-Fold ?�습 ?�료 ??�?Fold??결과 그래?��? ?�인?�고 ?�능???�약?�니??

In [ ]:
# ============================================================
# ??K-Fold ?�습 결과 ?�각??(?�택)
# ============================================================
from IPython.display import Image, display
from pathlib import Path
import yaml, sys
sys.path.insert(0, "src")
from utils import load_config, get_paths

cfg = load_config("config.yaml")
paths = get_paths(cfg)
kf_cfg = cfg.get("kfold", {"k": 5})
K = kf_cfg.get("k", 5)

plots = [
    ("results.png",          "?�� ?�습 결과 (Loss & Metrics)"),
    ("confusion_matrix.png", "?�� ?�동 ?�렬 (Confusion Matrix)"),
    ("PR_curve.png",         "?�� PR 곡선"),
    ("F1_curve.png",         "?�� F1 곡선"),
]

# 가중치 ?�일 존재 ?��? ?�약
print("="*52)
print(f"  K-Fold 가중치 ?�황 (k={K})")
print("="*52)
completed_folds = []
for fold_num in range(1, K + 1):
    best_pt = paths["weights"] / f"best_fold_{fold_num}.pt"
    if best_pt.exists():
        size_mb = best_pt.stat().st_size / 1e6
        print(f"  Fold {fold_num}: ??best_fold_{fold_num}.pt ({size_mb:.1f} MB)")
        completed_folds.append(fold_num)
    else:
        print(f"  Fold {fold_num}: ??미완�?")
print("="*52)
print()

# Fold�?결과 ?��?지 출력
for fold_num in range(1, K + 1):
    fold_dir = paths["runs"] / "kfold" / f"fold_{fold_num}"
    if not fold_dir.exists():
        continue
    print(f"\n{'='*40}")
    print(f"  ?�� Fold {fold_num}/{K} 결과")
    print(f"{'='*40}")
    found = False
    for filename, title in plots:
        p = fold_dir / filename
        if p.exists():
            print(title)
            display(Image(str(p)))
            found = True
    if not found:
        print(f"  결과 ?��?지 ?�음: {fold_dir}")

if not any((paths["runs"] / "kfold" / f"fold_{i}").exists() for i in range(1, K + 1)):
    print(f"결과 ?�더 ?�음: {paths['runs'] / 'kfold'}")
    print("K-Fold ?�습???�료?��? ?�았거나 결과 경로�??�인?�세??")

In [ ]:
# Colab ?��????�동 종료 �??�라?�브 ?�기??보장
import time
from google.colab import drive, runtime

print("?�라?�브 ?�일 ?�기?��? ?�해 ?��?�?..")
time.sleep(10) # ?�일 ?�???�료�??�한 ?��?ndrive.flush_and_unmount() # 모델 가중치 ?�이 ?�라?�브???�전???�기?�되?�록 강제 ?�??nprint("?��??�을 �?종료?�니??")
time.sleep(3)
runtime.unassign()